# 🔬 03 - 因子研究: IC分析与分层收益

本 Notebook 演示：
1. IC (Information Coefficient) 分析
2. 因子分层收益分析
3. 因子中性化

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.aggregator import DataAggregator
from datetime import date
import pandas as pd

agg = DataAggregator()
my_stocks = ['600519.SH', '000001.SZ', '300750.SZ', '000858.SZ', '601318.SH',
             '600036.SH', '000333.SZ', '601888.SH', '002714.SZ', '600900.SH']
bars = agg.get_bars_batch(my_stocks, date(2025, 1, 1), date.today())
prices = bars['close'].unstack(level=1)
print(f'✅ 数据: {len(bars)} 条, {len(my_stocks)} 只股票')

## 1. IC 分析

In [ ]:
from src.strategy.factors.momentum import MomentumFactor
from src.research.factor_analysis.ic_analysis import ICAnalyzer

# 计算因子
mom = MomentumFactor(window=10)
fv = mom.compute(bars)

# 计算前向收益和IC
analyzer = ICAnalyzer(forward_period=1)
fr = analyzer.compute_forward_returns(prices)
ic = analyzer.compute_ic(fv, fr)
rank_ic = analyzer.compute_rank_ic(fv, fr)

summary = analyzer.ic_summary(ic, rank_ic)
print('📊 IC 统计汇总:')
for k, v in summary.to_dict().items():
    print(f'  {k}: {v}')

In [ ]:
analyzer.plot_ic(ic, title='动量因子(10日) IC时序图')

## 2. 分层收益分析

In [ ]:
from src.research.factor_analysis.quantile import QuantileAnalyzer

qa = QuantileAnalyzer(n_quantiles=3)
result = qa.analyze(fv, fr)

print('📊 分层收益分析:')
print(f'  Spread: {result.spread*100:.4f}%')
print(f'  Monotonicity: {result.monotonicity:.4f}')
print(f'\n各层平均日收益:')
print(result.quantile_returns.mean())

In [ ]:
qa.plot(result, title='动量因子分层收益')

## 3. 因子中性化

In [ ]:
from src.research.factor_analysis.neutralization import FactorNeutralizer
import numpy as np

neutralizer = FactorNeutralizer()

# 创建行业映射
industry_map = {
    '600519.SH': '食品饮料', '000001.SZ': '银行', '300750.SZ': '电气设备',
    '000858.SZ': '食品饮料', '601318.SH': '非银金融', '600036.SH': '银行',
    '000333.SZ': '家用电器', '601888.SH': '旅游餐饮', '002714.SZ': '农林牧渔',
    '600900.SH': '公用事业',
}
dates = fv.index.get_level_values(0).unique()
symbols = fv.index.get_level_values(1).unique()
idx = pd.MultiIndex.from_product([dates, symbols], names=['date', 'symbol'])
industry = pd.Series([industry_map.get(s, '综合') for s in symbols] * len(dates), index=idx)
mcap = pd.Series(np.random.lognormal(20, 1, len(idx)), index=idx)

neutral = neutralizer.neutralize(fv, industry_codes=industry, market_cap=mcap)

print('📊 中性化前后对比:')
print(f'  原始因子均值: {fv.mean().iloc[0]:.6f}')
print(f'  中性化后均值: {neutral.mean().iloc[0]:.6f}')